[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-eval.ipynb)

# Model Evaluation & Cross-Validation

*AIBits Academy · Machine Learning End To End · Model Assessment*

A model is only as good as your ability to measure it honestly — rigorous evaluation frameworks separate genuine generalisation from lucky overfitting.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **📋 Real-World Case Study — Credit Card Fraud Detection**
>
> In a classic European cardholder dataset (283,726 transactions after de-duplication, only 473 fraudulent — 0.167% positive class), a Logistic Regression scored a seemingly excellent **99.91% accuracy**. But its confusion matrix told the real story: recall on the fraud class was just 0.58, meaning 42% of actual fraud slipped through undetected — accuracy was rewarding the model for correctly ignoring the 99.8% of legitimate transactions, not for catching fraud. Switching to Random Forest lifted fraud-class F1 from 0.69 to 0.84 (precision 0.97, recall 0.74) at the same 99.9%+ accuracy. Applying SMOTE oversampling before XGBoost pushed recall further to 0.79 but *cost* precision (0.76), netting a slightly lower F1 (0.77) than the un-oversampled XGBoost (0.85) — a concrete demonstration that oversampling is a precision/recall trade, not an automatic win, and must be judged against the real business cost of a false decline versus a missed fraud.

## The Fundamental Problem

Training error is an optimistic estimate — the model was tuned on that data. We need honest estimates of **generalisation error** (performance on unseen data). The solution: hold out data the model never sees during training.

## Evaluation Strategies

| Strategy | How | Pros | Cons |
|---|---|---|---|
| Train/Test Split | 80/20 or 70/30 split | Fast, simple | High variance estimate; wastes data |
| k-Fold CV | k splits; train on k−1, test on 1; average | Lower variance, uses all data | k× training time |
| Stratified k-Fold | Preserves class ratio in each fold | Essential for imbalanced classes | Same as k-Fold |
| Leave-One-Out (LOO) | n folds, one sample per fold | Unbiased for small n | Very slow for large n |
| Nested CV | Outer loop: evaluation; Inner loop: hyperparameter tuning | Unbiased with tuning | k_outer × k_inner × training time |

## Classification Metrics Deep Dive

In [ ]:
import numpy as np
from sklearn.metrics import (confusion_matrix, classification_report,
    roc_auc_score, average_precision_score, f1_score)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# Simulate Axis Bank credit card default dataset (imbalanced)
X, y = make_classification(n_samples=5000, n_features=12, n_informative=6,
                           weights=[0.93, 0.07], random_state=42)
print(f"Class distribution: {np.bincount(y)} (93% non-default, 7% default)")

# 5-fold stratified CV with multiple metrics
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

results = cross_validate(rf, X, y, cv=cv, scoring=[
    'accuracy','f1','roc_auc','average_precision'], return_train_score=True)

for m in ['accuracy','f1','roc_auc','average_precision']:
    tr = results[f'train_{m}'].mean()
    te = results[f'test_{m}'].mean()
    print(f"  {m:20s}  Train={tr:.3f}  CV={te:.3f}")

## Cohen's Kappa — Accuracy That Accounts For Chance

The 97.1% CV accuracy above sounds impressive, but recall the classes are 93%/7% — a classifier that *always* predicts "no default" would already score ~93% accuracy while having learned nothing at all. **Cohen's Kappa (κ)** fixes this by measuring agreement between predictions and ground truth *after* subtracting out the agreement expected purely from chance:

$$\kappa = \dfrac{p_0 - p_e}{1-p_e} \qquad p_0 = \text{observed accuracy},\ p_e = \text{accuracy expected by chance given the class proportions}$$

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score

# Same simulated Axis Bank credit-default dataset as above (93%/7% split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     stratify=y, random_state=42)

# Baseline: always predict the majority class ("no default")
majority_pred = np.zeros_like(y_test)
print(f"Majority baseline:  Accuracy={accuracy_score(y_test, majority_pred):.3f}  "
      f"Kappa={cohen_kappa_score(y_test, majority_pred):.3f}")

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(f"Random Forest:      Accuracy={accuracy_score(y_test, y_pred):.3f}  "
      f"Kappa={cohen_kappa_score(y_test, y_pred):.3f}")

The majority-class baseline scores a deceptively high 92.6% accuracy but κ=0.000, correctly exposing that it has zero real skill — it agrees with the truth no more than chance would, given this class split. The tuned Random Forest looks only marginally better on accuracy (94.3%, a ~2-point gain) but its κ=0.402 tells the real story: a moderate, genuinely-learned improvement over chance that raw accuracy almost completely hides.

| κ range | Interpretation |
|---|---|
| < 0 | Worse than chance |
| 0.01 – 0.20 | Slight agreement |
| 0.21 – 0.40 | Fair agreement |
| 0.41 – 0.60 | Moderate agreement |
| 0.61 – 0.80 | Substantial agreement |
| 0.81 – 1.00 | Almost perfect agreement |

> **⚠ Why Not Just Use Accuracy?**
>
> On imbalanced data (like this 93/7 credit-default split), accuracy alone can hide a model that has learned nothing useful. Kappa — like the F1/ROC-AUC/Average-Precision metrics above — belongs in the standard toolkit for exactly this situation; always report at least one chance-corrected or imbalance-aware metric alongside accuracy.

## Regression Metrics

| Metric | Formula | Interpretation | Scale-sensitive? |
|---|---|---|---|
| MAE | (1/n) Σ\|yᵢ − ŷᵢ\| | Average absolute error in y units | Yes |
| RMSE | √(MSE) | Standard deviation of residuals | Yes |
| MAPE | (100/n) Σ\|yᵢ−ŷᵢ\|/yᵢ | % error; scale-free | No (but fails at y≈0) |
| R² | 1 − SS_res/SS_tot | Variance explained (1=perfect) | No |
| Huber loss | MSE for small errors, MAE for large | Robust to outliers | Yes |

## Hyperparameter Tuning — GridSearch vs RandomSearch vs Bayesian

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_leaf': randint(1, 20),
    'max_features': ['sqrt', 'log2', 0.5],
    'class_weight': ['balanced', None],
}
rs = RandomizedSearchCV(RandomForestClassifier(random_state=42),
    param_dist, n_iter=30, cv=3, scoring='roc_auc',
    random_state=42, n_jobs=-1)
rs.fit(X, y)
print(f"Best ROC-AUC: {rs.best_score_:.4f}")
print(f"Best params : {rs.best_params_}")

## Probability Calibration — Is predict_proba() Actually Trustworthy?

Every threshold-tuning decision on the Handling Imbalanced Data page assumed the model's predicted probabilities genuinely reflect real-world frequencies — that among all cases the model scores at 0.7, roughly 70% actually belong to the positive class. Many models violate this badly. SVMs and Naive Bayes are notorious for producing scores that rank correctly (good AUC) but are systematically over- or under-confident as actual probabilities.

$$\text{A model is } \textbf{well-calibrated} \text{ when, among all predictions with } \hat{y}\approx p,\text{ the true positive rate} \approx p,\text{ for every } p\in[0,1]$$

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import brier_score_loss

X, y = make_classification(n_samples=4000, n_features=10, weights=[0.8,0.2], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

# Naive Bayes' independence assumption often produces overconfident, poorly-calibrated probabilities
nb = GaussianNB().fit(X_tr, y_tr)
nb_probs = nb.predict_proba(X_te)[:,1]

# CalibratedClassifierCV wraps any classifier and rescales its outputs using held-out folds
nb_calibrated = CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=5).fit(X_tr, y_tr)
nb_cal_probs = nb_calibrated.predict_proba(X_te)[:,1]

# Brier score: mean squared error between predicted probability and actual outcome (lower = better calibrated)
print(f"Uncalibrated Brier score: {brier_score_loss(y_te, nb_probs):.4f}")
print(f"Calibrated   Brier score: {brier_score_loss(y_te, nb_cal_probs):.4f}")

# Reliability curve data: bucket predictions, compare mean predicted prob vs actual fraction positive
frac_pos, mean_pred = calibration_curve(y_te, nb_probs, n_bins=10)
for mp, fp in zip(mean_pred, frac_pos):
    print(f"  predicted≈{mp:.2f}   actual fraction positive={fp:.2f}   gap={abs(mp-fp):.2f}")

Naive Bayes is systematically **under-confident** here — when it says 0.58, the true rate is actually 0.71. Any downstream threshold or cost-based decision built on the raw probabilities would be quietly miscalibrated. `CalibratedClassifierCV` fixes this by fitting a secondary mapping (Platt scaling / sigmoid, or isotonic regression for more flexibility with enough data) from raw scores to corrected probabilities, using held-out folds so the correction itself doesn't leak.

| Model type | Typical calibration behaviour |
|---|---|
| Logistic Regression | Usually well-calibrated by construction (it directly optimises log-loss) |
| Naive Bayes | Often overconfident or skewed due to the independence assumption |
| SVM (decision_function → sigmoid) | Scores are not probabilities at all until calibrated — always calibrate before using SVM probabilities for thresholding |
| Random Forest / Boosted Trees | Can be under-confident at the extremes (predictions pulled toward 0.5 by averaging) |

## Beyond the Test Set: Data & Concept Drift

Every evaluation technique so far assumes the test set is representative of what the model will see in production. That assumption quietly expires the moment real-world conditions shift — a model validated carefully today can silently degrade months later with no code change at all. Two distinct failure modes:

| Drift type | What changes | Example |
|---|---|---|
| **Data drift** (covariate shift) | The distribution of input features P(X) changes, but the true relationship P(y\|X) stays the same | A Flipkart pricing model trained pre-festival-season sees a very different mix of product categories once Diwali sales begin |
| **Concept drift** | The relationship P(y\|X) itself changes — the same inputs now genuinely predict a different outcome | A credit-risk model's "safe" income/EMI-ratio combinations shift after a macroeconomic shock changes what "risky" actually means |

In [ ]:
from scipy.stats import ks_2samp
import numpy as np

# Compare a feature's distribution: training-time snapshot vs. this week's live traffic
np.random.seed(14)
training_income = np.random.normal(45, 15, 2000)     # distribution the model was trained/validated on
live_income = np.random.normal(52, 18, 500)         # this week's incoming applicants — has the mix shifted?

# Kolmogorov-Smirnov test: are these two samples plausibly from the same distribution?
stat, p_value = ks_2samp(training_income, live_income)
print(f"KS statistic: {stat:.3f}   p-value: {p_value:.4f}")
if p_value < 0.01:
    print("Significant drift detected — investigate before trusting current predictions")

A statistically significant drift signal doesn't automatically mean the model is now wrong — but it's a prompt to check whether performance (not just the input distribution) has actually degraded, using any labels that have since become available, and to consider retraining on more recent data if it has. Running this kind of distributional comparison on a schedule (weekly/monthly, per key feature) is the standard lightweight monitoring practice that catches silent degradation before it shows up as a business problem.

## Statistically Comparing Models — Is the Difference Real?

> **📊 Prerequisite refresher**
>
> p-values, the null hypothesis, and what "statistically significant" actually claims (and doesn't claim) are covered in full on the **Inferential Statistics** prerequisite page — worth a quick look back if the framework below feels unfamiliar, since everything here is a direct application of the same six-step hypothesis-testing skeleton.

Every model comparison in this course so far has reported a mean CV score and quietly assumed the higher number is genuinely better. With only 5–10 folds, that gap can easily be noise. A paired significance test on the per-fold scores answers the real question directly:

In [ ]:
from scipy import stats
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores_lr = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=cv, scoring='roc_auc')
scores_rf = cross_val_score(RandomForestClassifier(n_estimators=200), X, y, cv=cv, scoring='roc_auc')

print(f"LR mean AUC: {scores_lr.mean():.4f}   RF mean AUC: {scores_rf.mean():.4f}")

# Naive paired t-test — treats the 10 fold-differences as independent samples
t_stat, p_naive = stats.ttest_rel(scores_rf, scores_lr)
print(f"Naive paired t-test:  t={t_stat:.4f}  p={p_naive:.4f}")

# Wilcoxon signed-rank — non-parametric alternative, no normality assumption
w_stat, p_wilcoxon = stats.wilcoxon(scores_rf, scores_lr)
print(f"Wilcoxon signed-rank: W={w_stat:.4f}  p={p_wilcoxon:.4f}")

Both tests agree Random Forest's 9.5-point AUC advantage here is real, not noise. But the naive paired t-test has a subtle flaw specific to cross-validation: the 10 folds are **not independent samples** — they share overlapping training data (each fold's training set is 90% of the same pool), which understates the true variance and can overstate significance. The **Nadeau & Bengio correction** adjusts for this directly:

In [ ]:
import numpy as np
from scipy.stats import t as tdist

diffs = scores_rf - scores_lr
n = len(diffs)
n_test_ratio, n_train_ratio = 1/10, 9/10          # 1 fold held out of 10 each time
corrected_var = diffs.var(ddof=1) * (1/n + n_test_ratio/n_train_ratio)
t_corrected = diffs.mean() / np.sqrt(corrected_var)
p_corrected = 2 * (1 - tdist.cdf(abs(t_corrected), df=n-1))
print(f"Corrected (Nadeau-Bengio) t-test: t={t_corrected:.4f}  p={p_corrected:.4f}")

The corrected p-value (0.0005) is larger than the naive one (effectively 0) — exactly the expected direction of the correction, since it accounts for the folds' shared training data inflating the naive test's confidence. The conclusion doesn't flip here (both agree the difference is significant), but on a closer comparison between two similarly-performing models, the naive test can claim significance where the corrected test would not — always prefer the corrected version, or the non-parametric Wilcoxon test, when reporting whether a CV score difference is genuinely reliable.

> **🔗 Real-World Link — A/B Testing Analysis**
>
> 294,478 real website visitors, split into a landing-page test. The honest result after cleaning the data: no statistically significant difference (p=0.19) — a genuine lesson that not every A/B test finds a winner. [See the case study →](https://statso.io/a-b-testing-case-study/) ·

## User Aspects of ML — Defining the Problem, Improving the Model, and Earning Trust

Every technique on this page assumes the problem was framed correctly and the model is worth improving in the first place. Three practical judgement calls that sit upstream of any metric:

| Question | Why it matters before touching any algorithm |
|---|---|
| What exactly is being predicted, at what unit of analysis? | "Predict customer churn" is ambiguous until it's pinned to a concrete definition (no purchase in 90 days? explicit cancellation?) — different definitions can produce entirely different labels from the same raw data |
| What's the real cost of being wrong, in each direction? | Determines which metric (recall-heavy vs. precision-heavy, as on the Handling Imbalanced Data page) actually reflects business priorities, before any model is even trained |
| Is the training population the same as the deployment population? | A model can be excellent on its test set and still fail in production if who it will see differs from who it was trained on (see the Ethics in ML page) |

When a model underperforms, the instinct is often "try a fancier algorithm" — but that's rarely the highest-leverage fix. A rough decision order, most reliable first:

1. **Check for label errors or leakage first.** No amount of algorithm tuning fixes a mislabeled training set or a feature that accidentally encodes the target.
2. **Add more informative features** (Feature Engineering page) before adding more rows — a genuinely predictive feature usually moves the needle more than doubling the dataset with the same feature set.
3. **Add more data**, specifically for the segments the model gets wrong most — not just more of the same easy majority-class examples.
4. **Only then, try a different model family** or tune hyperparameters more aggressively — this is usually the smallest lever of the four, despite often being reached for first.

Sometimes more data genuinely isn't available — a rare disease, a newly-launched product with little history, a low-frequency fraud pattern. In that regime: lean harder on the imbalanced-data techniques already covered (SMOTE, class weighting), consider whether a simpler, lower-variance model generalizes better from limited data than a complex one, and be explicit that the model's confidence intervals (or, for a genuinely principled treatment, the Bayesian methods on the next Advanced Extra Topics page) should widen accordingly rather than pretending scarce data supports a precise estimate.

> **⚠ Can I Trust My Model? A Quick Self-Check**
>
> Before treating a model as production-ready, this course has already built every tool needed to answer "can I trust it": is the CV comparison against alternatives statistically significant (above), not just numerically higher? Can a rejected/flagged case be explained (Model Interpretability)? Does it perform equitably across groups (ML Fairness)? Are its probabilities calibrated, not just well-ranked (above)? Is there a plan to detect drift after deployment (above)? A model that hasn't been checked against all five is not yet trustworthy, regardless of how good its headline accuracy looks.

## The Expected Value Framework — From Metrics to Rupees

Every metric on this page so far — accuracy, F1, AUC, calibration — answers "how good is the model, statistically?" None of them answer the question a business stakeholder actually asks: "should we act on this model's predictions, and on how many customers?" The **Expected Value framework** (Provost & Fawcett) closes that gap by attaching a real cost or benefit to each cell of the confusion matrix, so a model's quality is measured in the same units the business already thinks in — rupees, not AUC points.

$$\mathrm{EV} = p(\mathrm{TP})\cdot b(\mathrm{TP}) + p(\mathrm{FP})\cdot c(\mathrm{FP}) + p(\mathrm{FN})\cdot b(\mathrm{FN}) + p(\mathrm{TN})\cdot b(\mathrm{TN})$$

where p(·) is the fraction of the population falling in that confusion-matrix cell, and b(·) / c(·) are the real business benefit or cost of that outcome. This is exactly the same confusion matrix from the Classification Metrics section above — the framework simply asks you to price each cell before deciding on a threshold, rather than picking a threshold first and hoping it's sensible.

> **📋 Worked Example — HDFC Bank Credit-Card-Upgrade Offer**
>
> A targeted-offer classifier scores 2,400 held-out customers, of whom 373 (15.5%) would actually accept a credit-card-upgrade offer if contacted. Contacting a customer costs ₹80 (call-centre + processing); a successful upgrade nets ₹1,200 profit. Missing a genuine responder (not contacting them) costs nothing extra, since no action was taken.

Cell 8 needs the model's predicted probabilities on the test set from the earlier split.

In [ ]:
probs = rf.predict_proba(X_test)[:, 1]   # rf, X_test come from the earlier train/test split cell

In [ ]:
from sklearn.metrics import confusion_matrix

benefit_tp = 1200   # profit from a correctly-targeted upgrade
cost_fp    = 80     # wasted contact cost on a non-responder

def expected_value(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total = len(y_true)
    return (tp/total)*benefit_tp + (fp/total)*(-cost_fp)

# Strategy A: contact every customer
ev_all = expected_value(y_test, np.ones_like(y_test))
# Strategy B: contact based on the model's ranked score, sweeping thresholds
best_ev, best_t = max((expected_value(y_test, (probs>=t).astype(int)), t) for t in np.linspace(0,1,101))

The naive strategy — contact everyone — is not free just because it looks simple: every wasted ₹80 contact on a non-responder is a real cost, and with 2,027 non-responders in the test set that adds up fast. The model-ranked strategy finds the threshold where the marginal responder captured no longer justifies the marginal false-positive cost, and that threshold is almost never "0.5" — here it's 0.08, reflecting how cheap a contact is relative to how valuable a conversion is. This is precisely why "just use a 0.5 cutoff" from the Handling Imbalanced Data page is really a special case of Expected Value reasoning, not a law of nature.

## Profit Curves — Rank Customers, Then Ask "How Many?"

Expected Value picks the best *threshold*. A **profit curve** answers a related but distinct operational question: if we rank every customer by predicted score and contact the top X%, how does total profit change as X grows? This is often more actionable than a probability threshold, because call-centre capacity is usually planned in "how many customers today," not "which probability cutoff."

| Top X% contacted | Responders caught (of 373) | Total profit |
|---|---|---|
| 5% | 96 | ₹113,280 |
| 10% | 163 | ₹189,440 |
| 20% | 281 | ₹321,280 |
| 30% | 325 | ₹358,400 |
| **40%** | **357** | **₹380,160 (peak)** |
| 50% | 366 | ₹372,480 |
| 70% | 369 | ₹337,920 |
| 100% | 373 | ₹285,440 |

Profit rises steeply at first — the model's highest-scored customers are disproportionately real responders — peaks around the top 40% (₹380,160, matching the Expected Value threshold search above almost exactly), then *declines* beyond that point. Past the peak, each additional 1% of customers contacted contains mostly non-responders, so the ₹80 wasted-contact cost outpaces the trickle of remaining ₹1,200 conversions. A profit curve makes this trade-off visible at a glance in a way a single accuracy number never could — and it directly tells an operations team "call the top 40% of the ranked list today," a concrete, schedulable instruction.

## Cumulative Response & Lift Curves — How Much Better Than Random?

A **cumulative response curve** plots what fraction of all true responders you capture as you contact an increasing fraction of the ranked population. A **lift curve** re-expresses the same information as a multiple over random targeting: lift of 3x at the top 20% means the model-ranked top-20% contains 3 times as many responders as a random 20% sample would.

| Top X% contacted | % of all responders captured | Lift vs. random |
|---|---|---|
| 5% | 25.7% | **5.15x** |
| 10% | 43.7% | 4.37x |
| 20% | 75.3% | 3.77x |
| 30% | 87.1% | 2.90x |
| 50% | 98.1% | 1.96x |
| 100% | 100.0% | 1.00x |

Lift is always highest at the very top of the ranked list (5.15x here) and decays toward 1.0x as you contact the whole population — by definition, contacting everyone captures 100% of responders with zero lift over random, since "random" and "everyone" are the same strategy at X=100%. Marketing teams often report lift-at-top-decile (here, roughly 4.4x at the top 10%) as a single headline number for how much better a model is than blind outreach — but as the profit curve above shows, the *most profitable* cutoff (top 40%) is not the same as the *highest-lift* cutoff (top 5%). Lift answers "how good is the ranking," profit curves answer "where should we actually draw the line" — both are needed, and neither replaces the other.

> **💡 Going Deeper**
>
> This is only a taste of hyperparameter search. The next chapter, **Hyperparameter Tuning**, covers the Grid vs Random Search trade-off in depth, Successive Halving, and Bayesian Optimization with Optuna — plus why the nested CV idea introduced above is essential once you start tuning seriously.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Precision, recall and F1 by hand

From the true labels and predictions below, compute `tp`, `fp`, `fn`, then `precision`, `recall` and `f1` without scikit-learn's metric functions.

In [ ]:
import numpy as np
y_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0])
tp = fp = fn = precision = recall = f1 = None   # TODO


In [ ]:
try:
    check("counts", (tp, fp, fn) == (2, 1, 2))
    check("precision", round(precision, 4) == 0.6667)
    check("recall", recall == 0.5)
    check("f1", round(f1, 4) == 0.5714)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
y_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0])
tp = int(((y_pred == 1) & (y_true == 1)).sum())
fp = int(((y_pred == 1) & (y_true == 0)).sum())
fn = int(((y_pred == 0) & (y_true == 1)).sum())
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

```

</details>

### Exercise 2 · Medium · Stratified folds keep the class ratio

With a 5% positive rate, plain shuffling can produce folds with very different ratios. Use `StratifiedKFold(5, shuffle=True, random_state=0)` and store the positive rate of each **test** fold in `ratios`.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
rng = np.random.default_rng(0)
yy = (rng.random(1000) < 0.05).astype(int)
XX = np.zeros((1000, 1))
ratios = None   # TODO


In [ ]:
try:
    check("five folds", len(ratios) == 5)
    check("every fold is near the overall rate", all(abs(r - yy.mean()) < 0.005 for r in ratios))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.model_selection import StratifiedKFold
rng = np.random.default_rng(0)
yy = (rng.random(1000) < 0.05).astype(int)
XX = np.zeros((1000, 1))
ratios = [yy[te].mean() for _, te in StratifiedKFold(5, shuffle=True, random_state=0).split(XX, yy)]

```

</details>

### Exercise 3 · Stretch · ROC-AUC flatters imbalanced data

Fit a logistic regression on the imbalanced data and compute both `roc_auc_score` and `average_precision_score` (area under the precision-recall curve) on the test split. Store them in `roc` and `pr`; `pr_is_harsher` should say whether `pr < roc`.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
Xi, yi = make_classification(n_samples=8000, n_features=10, n_informative=4, weights=[0.97, 0.03], class_sep=0.7, random_state=3)
Xa, Xb, ya, yb = train_test_split(Xi, yi, stratify=yi, random_state=3)
roc = pr = pr_is_harsher = None   # TODO


In [ ]:
try:
    check("ROC-AUC looks fine", roc > 0.75)
    check("PR-AUC is much lower", pr_is_harsher is True and pr < roc - 0.15)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
Xi, yi = make_classification(n_samples=8000, n_features=10, n_informative=4, weights=[0.97, 0.03], class_sep=0.7, random_state=3)
Xa, Xb, ya, yb = train_test_split(Xi, yi, stratify=yi, random_state=3)
p = LogisticRegression(max_iter=1000).fit(Xa, ya).predict_proba(Xb)[:, 1]
roc, pr = roc_auc_score(yb, p), average_precision_score(yb, p)
pr_is_harsher = bool(pr < roc)

```

ROC-AUC counts the many easy negatives as wins; PR-AUC only rewards finding the rare positives, so it is the honest metric for fraud-style problems.

</details>

---
*Back to the course: **Machine Learning End To End → Model Evaluation & Cross-Validation**.*